In [17]:
import pandas as pd
import pandas as pd
import tensorflow as tf

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score

In [11]:
df = pd.read_csv('full_churn_dataset.csv')

In [12]:
# Remove missing values
df = df.dropna()

In [13]:
# Drop unique identifiers that do not contribute to predictive power
if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

In [14]:
# Separate features and target variable
X = df.drop(columns=['Churn'])
y = df['Churn'].astype(int)  # Ensure target is integer type

In [15]:
# Identify feature types
categorical_features = ['Gender', 'Subscription Type', 'Contract Length']
numerical_features = [col for col in X.columns if col not in categorical_features]

In [19]:
# Define the preprocessing pipeline using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

In [28]:
# Execute transformation
X_processed = preprocessor.fit_transform(X)

In [30]:
# ==========================================
# NEW: RECONSTRUCT DATAFRAME & SAVE TO CSV
# ==========================================

# Retrieve the newly generated column names for the one-hot encoded features
encoded_cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)

# Combine numerical and categorical column names to form the complete feature list
all_feature_names = list(numerical_features) + list(encoded_cat_features)

# Reconstruct the complete DataFrame (Features + Target)
processed_df = pd.DataFrame(X_processed, columns=all_feature_names)
processed_df['Churn'] = y.values

# Save the preprocessed dataset as a new CSV file
output_filename = 'preprocessed_churn_dataset.csv'
processed_df.to_csv(output_filename, index=False)
print(f"Success! Preprocessed dataset saved as: {output_filename}")

Success! Preprocessed dataset saved as: preprocessed_churn_dataset.csv


In [20]:
# Execute transformation and split data into training and validation sets
X_processed = preprocessor.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
input_dim = X_train.shape[1]

model = tf.keras.Sequential([
    # Input layer and first hidden layer with Batch Normalization and Dropout
    tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    # Second hidden layer
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    
    # Output layer for binary classification
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Users\User\ODL_Labs\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
# Compile model using the Adam optimizer and Binary Crossentropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [23]:
# Define callbacks for training efficiency and optimization
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=3, 
        min_lr=1e-6
    )
]

In [24]:
# Fit the neural network model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=512,  # Optimized for large-scale tabular datasets (~500k rows)
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8756 - auc: 0.9237 - loss: 0.3364 - val_accuracy: 0.9115 - val_auc: 0.9432 - val_loss: 0.2579 - learning_rate: 0.0010
Epoch 2/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9137 - auc: 0.9409 - loss: 0.2600 - val_accuracy: 0.9242 - val_auc: 0.9470 - val_loss: 0.2299 - learning_rate: 0.0010
Epoch 3/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9252 - auc: 0.9447 - loss: 0.2329 - val_accuracy: 0.9298 - val_auc: 0.9492 - val_loss: 0.2143 - learning_rate: 0.0010
Epoch 4/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9292 - auc: 0.9469 - loss: 0.2219 - val_accuracy: 0.9306 - val_auc: 0.9505 - val_loss: 0.2097 - learning_rate: 0.0010
Epoch 5/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9302 - auc: 0.9480 - loss: 0.2174 - val_accuracy: 0.9317 - val_auc: 0.9508 - val_loss: 0.2062 - learning_rate: 0.0010
Epoch 6/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9306 - auc: 0.

In [25]:
# Evaluate model performance on the validation partition
val_loss, val_accuracy, val_auc = model.evaluate(X_val, y_val, verbose=0)
print("\n=== Validation Set Metrics ===")
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"AUC-ROC:  {val_auc:.4f}\n")


=== Validation Set Metrics ===
Loss:     0.1925
Accuracy: 0.9333
AUC-ROC:  0.9526



In [26]:
# Generate probability predictions and class classifications
y_pred_prob = model.predict(X_val, verbose=0)
y_pred_class = (y_pred_prob > 0.5).astype(int)

In [27]:
# Display comprehensive classification report
print("=== Classification Report ===")
print(classification_report(y_val, y_pred_class))

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     44943
           1       0.90      0.99      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.94      0.93      0.93    101042
weighted avg       0.94      0.93      0.93    101042

